In [ ]:
!pip install pillow

In [ ]:
#!/usr/bin/env python3
"""
Crossword Generator
A sophisticated crossword puzzle generator with customizable layout and multiple output formats.

Author: aslepenkov
Version: 2.1.0
Date: 2026-04-13
"""

import csv
import os
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Union
from dataclasses import dataclass
from PIL import Image, ImageDraw, ImageFont

# ======================
# CONFIGURATION
# ======================

@dataclass
class CrosswordConfig:
    """Configuration settings for crossword generation."""
    
    # Grid settings
    grid_size: int = 50
    min_word_length: int = 3
    max_placement_attempts: int = 1000
    
    # Output settings
    output_dir: str = "crossword_output"
    filename_prefix: str = "crossword"
    
    # Image settings
    cell_size: int = 40
    font_size_letter: int = 24
    font_size_number: int = 14
    grid_color: str = "black"
    background_color: str = "white"
    text_color: str = "black"
    
    # File format settings
    save_png: bool = True
    save_csv: bool = True
    save_txt: bool = True
    
    # Font paths (in order of preference)
    font_paths: List[str] = None
    
    def __post_init__(self):
        """Initialize default font paths if not provided."""
        if self.font_paths is None:
            self.font_paths = [
                "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",  # Linux
                "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",  # Linux
                "/System/Library/Fonts/Arial.ttf",  # macOS
                "C:\\Windows\\Fonts\\arial.ttf",  # Windows
                "./arial.ttf"  # Local file
            ]

# ======================
# WORD DICTIONARY
# ======================

AI_ML_WORD_DICTIONARY = {
    "LLM": "Large language model that generates and understands text using neural networks",
    "PROMPT": "Input instruction given to a model to guide its output",
    "TOKEN": "Small unit of text used for processing by language models",
    "EMBEDDING": "Vector representation of text capturing semantic meaning",
    "FINE_TUNING": "Further training of a pretrained model on specific data",
    "RLHF": "Reinforcement learning from human feedback to align model behavior",
    "HALLUCINATION": "When a model generates incorrect or fabricated information",
    "CONTEXT_WINDOW": "Maximum amount of text a model can process at once",
    "INFERENCE": "Process of generating outputs from a trained model",
    "TRAINING": "Phase where a model learns patterns from data",
    "DATASET": "Collection of data used to train or evaluate models",
    "TRANSFORMER": "Neural network architecture used in modern LLMs",
    "ATTENTION": "Mechanism that weighs importance of different input tokens",
    "LOGITS": "Raw model outputs before probability normalization",
    "SAMPLING": "Method of selecting tokens during text generation",
    "TEMPERATURE": "Parameter controlling randomness of model output",
    "VECTOR_DATABASE": "Database optimized for storing and searching embeddings",
    "RAG": "Retrieval augmented generation combining search and LLM output",
    "AGENT": "System that uses an LLM to plan and execute multi-step tasks",
    "TOOL_USE": "Capability of a model to call external functions or APIs",
    "FUNCTION_CALLING": "Structured way for LLMs to invoke tools with arguments",
    "SYSTEM_PROMPT": "Hidden instruction that defines model behavior and constraints",
    "JAILBREAK": "Attempt to bypass model safety and usage restrictions",
    "ALIGNMENT": "Process of making model outputs consistent with human intent",
    "LATENCY": "Time delay between input and model response"
}

# ======================
# CROSSWORD GENERATOR CLASS
# ======================

size = 50
grid = [[None]*size for _ in range(size)]

# first word
first = max(words.keys(), key=len)
start = size//2 - len(first)//2

for i,c in enumerate(first):
    grid[size//2][start+i] = c

placements = {first: (size//2, start, 'horizontal')}


def check_perpendicular_sequences(word, start_y, start_x, direction):
    """Check if placing word creates invalid perpendicular sequences"""
    for i in range(len(word)):
        if direction == 'horizontal':
            # Check vertical sequence at each position
            y, x = start_y, start_x + i
            vertical_seq = ""

            # Collect letters above and below
            for dy in range(-size, size):
                ny = y + dy
                if 0 <= ny < size:
                    if ny == y:
                        vertical_seq += word[i]  # The letter we're placing
                    elif grid[ny][x] is not None:
                        vertical_seq += grid[ny][x]
                    else:
                        if vertical_seq and len(vertical_seq) > 1:
                            if vertical_seq not in words:
                                return False
                        vertical_seq = ""

            # Check final sequence
            if vertical_seq and len(vertical_seq) > 1:
                if vertical_seq not in words:
                    return False

        else:  # vertical
            # Check horizontal sequence at each position
            y, x = start_y + i, start_x
            horizontal_seq = ""

            # Collect letters left and right
            for dx in range(-size, size):
                nx = x + dx
                if 0 <= nx < size:
                    if nx == x:
                        horizontal_seq += word[i]  # The letter we're placing
                    elif grid[y][nx] is not None:
                        horizontal_seq += grid[y][nx]
                    else:
                        if horizontal_seq and len(horizontal_seq) > 1:
                            if horizontal_seq not in words:
                                return False
                        horizontal_seq = ""

            # Check final sequence
            if horizontal_seq and len(horizontal_seq) > 1:
                if horizontal_seq not in words:
                    return False

    return True


def place_word(word):

    for y in range(size):
        for x in range(size):

            if grid[y][x] is None:
                continue

            for i,c in enumerate(word):

                if c != grid[y][x]:
                    continue

                # horizontal
                startx = x-i

                if 0 <= startx and startx+len(word) < size:

                    ok=True
                    # Check word doesn't conflict with existing letters
                    for j,l in enumerate(word):
                        gx = startx+j
                        if grid[y][gx] not in (None,l):
                            ok=False

                    # Check space before and after word to prevent merging
                    if ok and startx > 0 and grid[y][startx-1] is not None:
                        ok=False  # No space before word
                    if ok and startx+len(word) < size-1 and grid[y][startx+len(word)] is not None:
                        ok=False  # No space after word

                    # Check perpendicular sequences don't create invalid words
                    if ok and not check_perpendicular_sequences(word, y, startx, 'horizontal'):
                        ok=False

                    if ok:
                        for j,l in enumerate(word):
                            grid[y][startx+j]=l
                        return (y, startx, 'horizontal')

                # vertical
                starty = y-i

                if 0 <= starty and starty+len(word) < size:

                    ok=True
                    # Check word doesn't conflict with existing letters
                    for j,l in enumerate(word):
                        gy = starty+j
                        if grid[gy][x] not in (None,l):
                            ok=False

                    # Check space above and below word to prevent merging
                    if ok and starty > 0 and grid[starty-1][x] is not None:
                        ok=False  # No space above word
                    if ok and starty+len(word) < size-1 and grid[starty+len(word)][x] is not None:
                        ok=False  # No space below word

                    # Check perpendicular sequences don't create invalid words
                    if ok and not check_perpendicular_sequences(word, starty, x, 'vertical'):
                        ok=False

                    if ok:
                        for j,l in enumerate(word):
                            grid[starty+j][x]=l
                        return (starty, x, 'vertical')

    return None


# Sort words by length (longest first) and try to place them
sorted_words = sorted(words.keys(), key=len, reverse=True)
for w in sorted_words:
    if w == first:
        continue
    placement = place_word(w)
    if placement:
        placements[w] = placement


# ======================
# Numbering and clues
# ======================

numbering = {}
next_number = 1

for y in range(size):
    for x in range(size):
        if grid[y][x] is None:
            continue

        starts_across = (x == 0 or grid[y][x-1] is None) and (x + 1 < size and grid[y][x+1] is not None)
        starts_down = (y == 0 or grid[y-1][x] is None) and (y + 1 < size and grid[y+1][x] is not None)

        if starts_across or starts_down:
            numbering[(y, x)] = next_number
            next_number += 1

across_clues = []
down_clues = []

for word, (y, x, direction) in placements.items():
    number = numbering.get((y, x))
    if number is None:
        continue
    clue_text = words[word]
    if direction == 'horizontal':
        across_clues.append((number, word, clue_text))
    else:
        down_clues.append((number, word, clue_text))

across_clues.sort(key=lambda item: item[0])
down_clues.sort(key=lambda item: item[0])

print("\nCROSSWORD (ASCII2)\n")
for row in grid:
    line = ""
    for cell in row:
        if cell:
            line += cell + " "
        else:
            line += "_ "
    print(line)

print("\nGRID NUMBERING\n")
for y in range(size):
    line = ""
    for x in range(size):
        if grid[y][x] is None:
            line += "__ "
        elif (y, x) in numbering:
            line += f"{numbering[(y, x)]:02d} "
        else:
            line += ".. "
    print(line)

print("\nTEXT CLUES\n")
print("Across:")
for number, word, clue_text in across_clues:
    print(f"{number}. {clue_text} ({len(word)} letters)")

print("\nDown:")
for number, word, clue_text in down_clues:
    print(f"{number}. {clue_text} ({len(word)} letters)")

text_lines = []
text_lines.append("GRID NUMBERING")
for y in range(size):
    line = ""
    for x in range(size):
        if grid[y][x] is None:
            line += "__ "
        elif (y, x) in numbering:
            line += f"{numbering[(y, x)]:02d} "
        else:
            line += ".. "
    text_lines.append(line)

text_lines.append("")
text_lines.append("TEXT CLUES")
text_lines.append("Across:")
for number, word, clue_text in across_clues:
    text_lines.append(f"{number}. {clue_text} ({len(word)} letters)")

text_lines.append("")
text_lines.append("Down:")
for number, word, clue_text in down_clues:
    text_lines.append(f"{number}. {clue_text} ({len(word)} letters)")

with open('crossword_text.txt', 'w', encoding='utf-8') as textfile:
    textfile.write("\n".join(text_lines))

print("\nCreated file crossword_text.txt")


# ======================
# CSV output
# ======================

with open('crossword_answers.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    for row in grid:
        csv_row = [cell if cell else '' for cell in row]
        writer.writerow(csv_row)

with open('crossword_blank.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    for row in grid:
        csv_row = ['' if cell else '' for cell in row]
        writer.writerow(csv_row)

print("\nCreated files crossword_answers.csv, crossword_blank.csv")


# ======================
# PNG
# ======================

cell = 40
img_answers = Image.new("RGB", (size*cell, size*cell), "white")
draw_answers = ImageDraw.Draw(img_answers)
img_blank = Image.new("RGB", (size*cell, size*cell), "white")
draw_blank = ImageDraw.Draw(img_blank)

try:
    # Try to load system fonts
    import os
    font_paths = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",  # Linux
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",  # Linux
        "/System/Library/Fonts/Arial.ttf",  # macOS
        "C:\\Windows\\Fonts\\arial.ttf",    # Windows
        "./arial.ttf"  # Local file
    ]

    letter_font = None
    number_font = None
    for path in font_paths:
        if os.path.exists(path):
            letter_font = ImageFont.truetype(path, 24)
            number_font = ImageFont.truetype(path, 14)
            break

    if letter_font is None:
        letter_font = ImageFont.load_default()
        number_font = ImageFont.load_default()

except:
    letter_font = ImageFont.load_default()
    number_font = ImageFont.load_default()

for y in range(size):
    for x in range(size):

        x1 = x * cell
        y1 = y * cell
        x2 = x1 + cell
        y2 = y1 + cell

        draw_answers.rectangle([x1, y1, x2, y2], outline="black")

        if (y, x) in numbering:
            num_text = str(numbering[(y, x)])
            bbox_num = draw_answers.textbbox((0, 0), num_text, font=number_font)
            w_num = bbox_num[2] - bbox_num[0]
            h_num = bbox_num[3] - bbox_num[1]
            draw_answers.text((x1 + 4, y1 + 2), num_text, fill="black", font=number_font)
            draw_blank.text((x1 + 4, y1 + 2), num_text, fill="black", font=number_font)

        if grid[y][x]:
            draw_blank.rectangle([x1, y1, x2, y2], outline="black", width=2)
            letter = grid[y][x]
            bbox_letter = draw_answers.textbbox((0, 0), letter, font=letter_font)
            w_letter = bbox_letter[2] - bbox_letter[0]
            h_letter = bbox_letter[3] - bbox_letter[1]

            draw_answers.text(
                (x1 + (cell - w_letter) / 2, y1 + (cell - h_letter) / 2),
                letter,
                fill="black",
                font=letter_font
            )

img_answers.save("crossword_answers.png")
img_blank.save("crossword_blank.png")

print("\nCreated files: crossword_answers.png, crossword_blank.png")

from IPython.display import Image, display
display(Image('crossword_answers.png'))
display(Image('crossword_blank.png'))
